In [6]:
import pandas as pd
import re
from collections import Counter

# ==========================================
# 1. LOAD EXCEL DATASET
# ==========================================

df = pd.read_excel("search_query_spelling_corrector_210.xlsx")

print("Dataset loaded:", df.shape)


# ==========================================
# 2. CREATE SPELLING DICTIONARY
# ==========================================

spelling_dictionary = {}

for i in range(len(df)):

    wrong = df.loc[i, "Misspelled_Query"].lower().split()
    correct = df.loc[i, "Correct_Query"].lower().split()

    for w, c in zip(wrong, correct):
        spelling_dictionary[w] = c

print("Known spelling corrections:")
print(spelling_dictionary)


# ==========================================
# 3. CREATE VOCABULARY
# ==========================================

vocabulary = []

for query in df["Correct_Query"]:

    words = re.findall(
        r'\b[a-zA-Z]+\b',
        query.lower()
    )

    vocabulary.extend(words)

word_frequency = Counter(vocabulary)

vocabulary = set(vocabulary)


# ==========================================
# 4. LEVENSHTEIN DISTANCE
# ==========================================

def levenshtein_distance(word1, word2):

    rows = len(word1) + 1
    cols = len(word2) + 1

    matrix = [
        [0] * cols
        for _ in range(rows)
    ]

    for i in range(rows):
        matrix[i][0] = i

    for j in range(cols):
        matrix[0][j] = j

    for i in range(1, rows):

        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,
                matrix[i][j - 1] + 1,
                matrix[i - 1][j - 1] + cost
            )

    return matrix[-1][-1]


# ==========================================
# 5. FIND CORRECT WORD
# ==========================================

def correct_word(word):

    word = word.lower()

    # First check our spelling dictionary
    if word in spelling_dictionary:
        return spelling_dictionary[word]

    # Already known correct word
    if word in vocabulary:
        return word

    best_word = word
    best_distance = 999

    # Only compare with words of similar length
    for candidate in vocabulary:

        if abs(len(word) - len(candidate)) > 2:
            continue

        distance = levenshtein_distance(
            word,
            candidate
        )

        if distance < best_distance:

            best_distance = distance
            best_word = candidate

        elif distance == best_distance:

            if word_frequency[candidate] > word_frequency[best_word]:
                best_word = candidate

    # Don't make a correction if the word is too different
    if best_distance <= 2:
        return best_word

    return word


# ==========================================
# 6. CORRECT COMPLETE QUERY
# ==========================================

def correct_query(query):

    words = query.lower().split()

    corrected_words = []

    for word in words:

        clean_word = re.sub(
            r'[^a-zA-Z]',
            '',
            word
        )

        if clean_word == "":
            corrected_words.append(word)
            continue

        corrected = correct_word(clean_word)

        corrected_words.append(corrected)

    return " ".join(corrected_words)


# ==========================================
# 7. TEST DATASET
# ==========================================

print("\n================================")
print("DATASET TESTING")
print("================================")

for i in range(10):

    original = df.loc[i, "Misspelled_Query"]
    actual = df.loc[i, "Correct_Query"]

    predicted = correct_query(original)

    print("\nOriginal :", original)
    print("Actual   :", actual)
    print("Predicted:", predicted)


# ==========================================
# 8. NEW USER QUERY
# ==========================================

print("\n================================")
print("SEARCH QUERY SPELLING CORRECTOR")
print("================================")

user_query = input(
    "\nEnter a search query: "
)

result = correct_query(user_query)

print("\nOriginal Query:")
print(user_query)

print("\nCorrected Query:")
print(result)

Dataset loaded: (210, 3)
Known spelling corrections:
{'best': 'best', 'restarant': 'restaurant', 'course': 'course', 'learn': 'learn', 'online': 'online', 'how': 'how', 'to': 'to', 'tutorial': 'tutorial', 'free': 'free', 'latest': 'latest', 'information': 'information', 'near': 'near', 'me': 'me', 'lern': 'learn', 'tomorow': 'tomorrow', 'moblie': 'mobile', 'hospitel': 'hospital', 'tutoral': 'tutorial', 'restaurent': 'restaurant', 'corect': 'correct', 'lerning': 'learning', 'analaysis': 'analysis', 'artifical': 'artificial', 'collge': 'college', 'studnts': 'students', 'aply': 'apply', 'fligts': 'flights', 'laptp': 'laptop', 'pythn': 'python', 'javascrpt': 'javascript', 'programing': 'programming', 'scholrship': 'scholarship', 'intership': 'internship', 'wether': 'weather', 'corse': 'course', 'developr': 'developer', 'aplication': 'application', 'univercity': 'university', 'certifcate': 'certificate', 'analaytics': 'analytics', 'begginer': 'beginner', 'reciepe': 'recipe'}

DATASET TESTIN


Enter a search query:  best restarant course



Original Query:
best restarant course

Corrected Query:
best restaurant course
